# Closed channels in $\alpha$ + $^{12}$C scattering

A **closed channel** is one whose threshold is above the current beam energy. It still affects the solution inside the interaction region, but it does not appear as an outgoing asymptotic scattering channel. This notebook shows how to handle that situation with the public `lax.models` and `lax.spectral` helpers.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np

import lax as lm
from lax.boundary import BoundaryValues


def benchmark_data_dir() -> Path:
    search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    search_roots.append(Path(lm.__file__).resolve().parents[2])
    for root in search_roots:
        candidate = root / "tests" / "benchmarks" / "data"
        if candidate.is_dir():
            return candidate
    msg = (
        "Could not locate tests/benchmarks/data from the current notebook environment."
    )
    raise FileNotFoundError(msg)


fixture = json.loads((benchmark_data_dir() / "descouvemont_alpha_c12.json").read_text())
reference = fixture["references"][2]

model = lm.models.ALPHA_C12_ROTOR_MODEL
channels = lm.models.channels_from_rotor_model(model)
energies = np.asarray(reference["energies"], dtype=np.float64)

[
    {
        "index": index,
        "label": channel.label,
        "threshold_mev": channel.threshold,
    }
    for index, channel in enumerate(model.channels)
]

## What the model object gives you

The public preset `lm.models.ALPHA_C12_ROTOR_MODEL` bundles together:

- the channel thresholds,
- the optical-model depths and geometry,
- the deformation parameters,
- and the projectile/target charges needed for the Coulomb term.

That means the notebook can stay focused on the workflow instead of re-deriving model constants cell by cell.


In [ ]:
def boundary_at_energy(boundary: BoundaryValues, energy_index: int) -> BoundaryValues:
    return BoundaryValues(
        H_plus=boundary.H_plus[energy_index],
        H_minus=boundary.H_minus[energy_index],
        H_plus_p=boundary.H_plus_p[energy_index],
        H_minus_p=boundary.H_minus_p[energy_index],
        is_open=boundary.is_open[energy_index],
        k=boundary.k[energy_index],
    )


solver = lm.compile(
    mesh=lm.MeshSpec(
        "legendre",
        "x",
        n=int(reference["n_basis"]),
        scale=float(reference["scale"]),
        extras={"n_intervals": int(reference["n_intervals"])},
    ),
    channels=channels,
    operators=("T+L", "1/r^2"),
    solvers=("rmatrix_direct",),
    energies=energies,
    method="linear_solve",
    V_is_complex=True,
    z1z2=(model.projectile_charge, model.target_charge),
)

interaction = lm.models.interaction_from_rotor_model(model, solver)
r_values = solver.rmatrix_direct(interaction)

rows = []
for energy_index, energy in enumerate(energies):
    boundary = boundary_at_energy(solver.boundary, energy_index)
    smatrix = np.asarray(
        lm.spectral.open_channel_smatrix_from_R(r_values[energy_index], boundary)
    )
    open_count = lm.models.open_channel_count(model, float(energy))
    amplitudes, phases = lm.models.first_column_amplitudes_and_phases(
        smatrix, open_count
    )
    rows.append(
        {
            "energy_mev": float(energy),
            "open_channels": open_count,
            "first_column_amplitudes": amplitudes.tolist(),
            "reference_amplitudes": reference["amplitudes"][energy_index],
            "first_column_phases": phases.tolist(),
            "reference_phases": reference["phases"][energy_index],
        }
    )

rows

## What the first column means

The first column of the open-channel `S` matrix answers a practical question:

> If the incoming wave starts in the elastic entrance channel, how much amplitude emerges in each open outgoing channel?

The amplitude tells you the size of that channel-to-channel coupling. The phase tells you how the outgoing wave is shifted relative to free propagation.


## How to adapt this notebook

Try changing just one thing at a time:

- replace the preset model with a modified copy that changes the deformation or optical depths,
- change `energies` to watch channels open one by one,
- or increase `n_basis` / `n_intervals` to study convergence.

The public helpers keep those experiments small and readable.
